# Глава 5. Статистика и проверка гипотез — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Вероятность: частота и симуляция

Статистика начинается с вопроса «а не случайно ли это?». Чтобы на него отвечать, нужен язык вероятностей. Без формул-заклинаний: вероятность — это доля в длинной серии, её можно посчитать по данным или проверить симуляцией — заставить компьютер «прожить» ситуацию тысячи раз.

Теория — на сайте курса: `ch05/01-probability.html`. Задачи «собери код из строк» (1) остались на сайте.

### Вероятность опоздания · разминка

Сохраните в `p_late` оценку вероятности опоздания по данным (время 1–299 минут), а в `p_on_time` — вероятность доставки вовремя. Обе — 3 знака.

*Подсказки (2): `dsc.hint('p-late')`*

In [ ]:
%%task ch05/01-probability/p-late
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()

### Две доставки · разминка

Гость заказал две доставки, каждая опаздывает с вероятностью 0.24, независимо. Посчитайте по формулам (3 знака): `both` — обе опоздали, `none` — ни одна не опоздала, `exactly_one` — ровно одна. Проверьте себя: сумма трёх вероятностей должна быть 1.

*Подсказки (2): `dsc.hint('two-late')`*

In [ ]:
%%task ch05/01-probability/two-late
p = 0.24

### Условная вероятность · разминка

Сохраните `p_late_walk` — P(опоздание | пеший курьер) и `p_walk_late` — P(пеший курьер | опоздание). Обе — 3 знака.

*Подсказки (2): `dsc.hint('cond')`*

In [ ]:
%%task ch05/01-probability/cond
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]

### Хотя бы одна · база

Гость заказывает доставку `n = 5` раз за две недели, вероятность опоздания каждой — `p = 0.24`, опоздания независимы. Сохраните в `p_formula` вероятность, что опоздает **хотя бы одна** (по формуле, 3 знака), а в `p_sim` — оценку симуляцией по 100 000 повторений (3 знака).

*Подсказки (2): `dsc.hint('at-least')`*

In [ ]:
%%task ch05/01-probability/at-least
import numpy as np
p, n = 0.24, 5
rng = np.random.default_rng(5)

### Независимы ли события · база

Проверьте, независимы ли опоздание и снег. Сохраните `p_joint` — P(опоздание и снег), `p_product` — P(опоздание) × P(снег) (обе — 4 знака) и `ratio` — их отношение (2 знака).

*Подсказки (2): `dsc.hint('independence')`*

In [ ]:
%%task ch05/01-probability/independence
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
snow = ok["weather"] == "снег"

### Снег в двух направлениях · посложнее

Постройте таблицу `t = pd.crosstab(ok["weather"] == "снег", ok["late"], margins=True)` и по её числам сохраните: `n_snow_late` — число опоздавших снежных доставок, `p_late_snow` — P(опоздание | снег) и `p_snow_late` — P(снег | опоздание) (3 знака), `extra` — сколько опозданий было бы в снег, если бы там опаздывали с общей частотой (число снежных доставок × P(опоздание), 1 знак), и `added` — сколько опозданий снег добавил сверх этого (`n_snow_late − extra`, 1 знак).

*Подсказки (2): `dsc.hint('snow-table')`*

In [ ]:
%%task ch05/01-probability/snow-table
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]

### Закон больших чисел · база

Смоделируйте 5 000 доставок с вероятностью опоздания 0.24 (`rng = np.random.default_rng(11)`), посчитайте накопленную долю опозданий `running` (массив той же длины) и нарисуйте её линией вместе с горизонтальной линией на 0.24. В `after_100` и `after_5000` сохраните отклонение доли от 0.24 (по модулю) после 100-й и после 5 000-й доставки.

*Подсказки (2): `dsc.hint('running')`*

In [ ]:
%%task ch05/01-probability/running
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(11)

### Почему маленькие группы шумят · база

Представьте 20 кофеен, у каждой ровно 30 доставок, и у **всех** настоящая вероятность опоздания одинаковая — 0.24. Смоделируйте долю опозданий каждой кофейни (`rng = np.random.default_rng(8)`, таблица 20 × 30) и сохраните `shares` (массив из 20 долей), `spread` — разницу между самой большой и самой маленькой долей (2 знака).

*Подсказки (2): `dsc.hint('small-groups')`*

In [ ]:
%%task ch05/01-probability/small-groups
import numpy as np
rng = np.random.default_rng(8)

### Восемь опозданий за смену · со звёздочкой

Курьер за смену делает 20 доставок, каждая опаздывает с вероятностью 0.24 независимо от других. Смоделируйте 200 000 смен (`rng = np.random.default_rng(21)`) и сохраните в `p_8plus` оценку вероятности, что за смену опоздают **8 и более** доставок (4 знака). В `typical` — самое частое число опозданий за смену.

*Подсказки (2): `dsc.hint('ch05/01-probability/shift')` · задача по желанию*

In [ ]:
%%task ch05/01-probability/shift
import numpy as np
rng = np.random.default_rng(21)

## Урок 2. Распределения: мат. ожидание, биномиальное, Пуассона

У случайности есть форма. Считаем мат. ожидание — средний результат «в долгую» — и знакомимся с двумя распределениями для счёта: биномиальным (опоздания за смену) и пуассоновским (гости за 5 минут). Всё — одной строкой scipy.stats вместо симуляции. Нормальное распределение — в следующем уроке.

Теория — на сайте курса: `ch05/02-distributions.html`.

### Ровно пять · разминка

Сохраните в `p5` вероятность, что из 20 доставок опоздает ровно 5 (p = 0.24), а в `p_le3` — что опоздает не больше 3. Обе — 4 знака.

*Подсказки (2): `dsc.hint('pmf')`*

In [ ]:
%%task ch05/02-distributions/pmf
from scipy import stats
n, p = 20, 0.24

### Плохая смена · разминка

Сохраните в `p_bad` вероятность, что за смену из 20 доставок опоздает **8 и больше** (p = 0.24), 4 знака.

*Подсказки (2): `dsc.hint('ch05/02-distributions/tail')`*

In [ ]:
%%task ch05/02-distributions/tail
from scipy import stats

### Десерты к заказу · разминка

По истории заказов число десертов в одном заказе распределено так: 0 десертов — 0.55, 1 — 0.35, 2 — 0.08, 3 — 0.02. Сохраните в `ev` мат. ожидание числа десертов на заказ (2 знака), а в `per_500` — сколько десертов в среднем уйдёт на 500 заказов (целое число, `round`).

*Подсказки (3): `dsc.hint('ev-toy')`*

In [ ]:
%%task ch05/02-distributions/ev-toy
import numpy as np
values = np.array([0, 1, 2, 3])
probs = np.array([0.55, 0.35, 0.08, 0.02])

### Среднее и разброс · база

За неделю курьерская служба делает 200 доставок, p = 0.24. Сохраните `mu` и `sigma` — среднее и стандартное отклонение числа опозданий за неделю (по формулам n·p и √(n·p·(1−p)), 2 знака), и `p60` — вероятность 60 и более опозданий за неделю (4 знака).

*Подсказки (2): `dsc.hint('binom-moments')`*

In [ ]:
%%task ch05/02-distributions/binom-moments
from scipy import stats
n, p = 200, 0.24

### Нарисовать распределение · база

Нарисуйте столбцами биномиальное распределение для n = 30, p = 0.24: по X — число опозданий от 0 до 30, по Y — вероятность. Отметьте среднее вертикальной линией.

*Подсказки (2): `dsc.hint('pmf-plot')`*

In [ ]:
%%task ch05/02-distributions/pmf-plot
from scipy import stats
import matplotlib.pyplot as plt
n, p = 30, 0.24

### Биномиальное руками · со звёздочкой

Повторите «Разберём вместе» для смены из **4** доставок (p = 0.24) и вопроса «ровно 2 опоздания». Сохраните: `ways` — сколько существует вариантов вроде «ООВВ» с ровно двумя «О»; `one` — вероятность одного такого варианта (4 знака); `p_hand` — `ways * one` (4 знака) и `p_scipy` — то же через `stats.binom.pmf` (4 знака). Число вариантов можно найти перебором в коде или выписать варианты на бумаге.

*Подсказки (2): `dsc.hint('binom-hand')` · задача по желанию*

In [ ]:
%%task ch05/02-distributions/binom-hand
from scipy import stats

### Сколько стоит купон с льготой · база

Отдел доставки смягчил правило: опоздание до 5 минут включительно — без купона, от 6 до 20 минут — купон 150 ₽, больше 20 минут — 400 ₽. Для доставок с временем 1–299 минут постройте столбец `coupon` по этому правилу. Сохраните `p_150` и `p_400` — доли доставок с купоном 150 и 400 ₽ (4 знака), `ev` — ожидание купона на доставку (2 знака) и `week` — ожидаемый расход за неделю из 200 доставок (целое число рублей).

*Подсказки (3): `dsc.hint('ev-comp')`*

In [ ]:
%%task ch05/02-distributions/ev-comp
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
late_min = ok["actual_min"] - ok["promised_min"]

### Очередь в Уфе · база

В кофейне UFA-1 в тихие часы один бариста справляется, если за 5 минут подходит не больше трёх гостей. По `arrivals.csv` для UFA-1 сохраните: `lam` — среднее число гостей за пятиминутку (3 знака), `ratio` — отношение дисперсии к среднему (2 знака), `p_queue` — вероятность 4 и больше гостей по распределению Пуассона с этим $\lambda$ (4 знака; можно от `lam` или от неокруглённого среднего — проверка примет оба), `share_queue` — та же доля по данным (4 знака) и `per_day` — сколько «очередных» пятиминуток ожидается за двухчасовое окно из 24 пятиминуток (по модели, 2 знака).

*Подсказки (3): `dsc.hint('poisson-queue')`*

In [ ]:
%%task ch05/02-distributions/poisson-queue
import pandas as pd
from scipy import stats
a = pd.read_csv("arrivals.csv")

## Урок 3. Нормальное распределение, приближение и z-оценка

Колокол — самое известное распределение: на нём стоят доверительные интервалы и тесты. Разбираемся, что значит «правило 68–95–99.7», когда колоколом можно заменить точный биномиальный расчёт, как z-оценка сравнивает несравнимое и почему время доставки и деньги колоколу не подчиняются.

Теория — на сайте курса: `ch05/02b-normal.html`. Задачи «собери код из строк» (1) остались на сайте.

### Квантиль нормального · база

Время сборки заказа нормально: среднее 12 минут, σ = 3. Сохраните `t90` — время, быстрее которого собирается 90 % заказов, и `p_9_15` — вероятность, что сборка займёт от 9 до 15 минут. Оба — 3 знака.

*Подсказки (2): `dsc.hint('ch05/02b-normal/quantile')`*

In [ ]:
%%task ch05/02b-normal/quantile
from scipy import stats

### Месяц большой кофейни · база

За месяц кофейня делает 500 доставок, p = 0.24. Сохраните `exact` — точную вероятность 140 и более опозданий (`binom`, 4 знака), `approx` — ту же вероятность по нормальному приближению с поправкой на непрерывность (4 знака), `rel_err` — относительную ошибку приближения `|approx − exact| / exact` (по неокруглённым числам, 3 знака) и `rule_ok` — `True`, если выполнено правило n·p ≥ 10 и n·(1 − p) ≥ 10 (посчитайте n·p и n·(1 − p) кодом из `n` и `p`).

*Подсказки (3): `dsc.hint('norm-approx')`*

In [ ]:
%%task ch05/02b-normal/norm-approx
import math
from scipy import stats
n, p, k = 500, 0.24, 140

### Самый загруженный день · база

Посчитайте число доставок по дням (`created` — дата) по всем строкам, без фильтра по времени доставки: здесь важно, сколько заказов пришло, а не сколько ехали. Найдите их среднее и стандартное отклонение. Сохраните `z_max` — z-оценку самого загруженного дня (2 знака) и `n_unusual` — сколько дней имеют |z| > 2.

*Подсказки (2): `dsc.hint('zday')`*

In [ ]:
%%task ch05/02b-normal/zday
import pandas as pd
d = pd.read_csv("deliveries.csv", parse_dates=["created"])

### Подозрительный пилот · база

Новая кофейня за первую неделю сделала 50 доставок, и 20 из них опоздали. Если бы она опаздывала так же, как сеть (p = 0.24), насколько редко такое случалось бы? Сохраните `expected` — ожидаемое число опозданий, `p_20plus` — вероятность 20 и более опозданий из 50 (4 знака) и `z` — z-оценку результата по формулам биномиального распределения (2 знака).

*Подсказки (2): `dsc.hint('pilot50')`*

In [ ]:
%%task ch05/02b-normal/pilot50
from scipy import stats
n, p, observed = 50, 0.24, 20

### Хвост суммы заказа · посложнее

В теории мы видели, что время доставки скошено вправо. Проверьте, так ли с деньгами: для суммы заказа `amount` посчитайте среднее и σ и сохраните `real_tail` — долю заказов дороже «среднее + 2σ» по данным, `normal_tail` — сколько таких было бы при нормальном распределении (`stats.norm.sf(2)`), обе 4 знака, и `ratio` — во сколько раз данные «тяжелее» модели (`real_tail / normal_tail`, 1 знак). Нарисуйте гистограмму `amount` (`density=True`) с нормальной кривой поверх.

*Подсказки (2): `dsc.hint('tails')`*

In [ ]:
%%task ch05/02b-normal/tails
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

d = pd.read_csv("deliveries.csv")
a = d["amount"]

### Логарифм выпрямляет хвост · со звёздочкой

Сравните асимметрию (`skew()`) времени доставки (1–299 минут) и его натурального логарифма (`np.log`). Сохраните `skew_raw` и `skew_log` (2 знака). Нарисуйте две гистограммы рядом: слева — время, справа — логарифм, заголовки «минуты» и «log(минуты)».

*Подсказки (2): `dsc.hint('ch05/02b-normal/log')` · задача по желанию*

In [ ]:
%%task ch05/02b-normal/log
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"]

## Урок 4. Выборки и центральная предельная теорема

Мы почти никогда не видим всех данных — только выборку: доставки за три месяца, а не за всё время; 6 000 гостей в тесте, а не всех будущих. Разбираемся, насколько среднее по выборке может отличаться от настоящего, почему средние распределены нормально, как от размера выборки зависит точность — и как выборка, собранная «как удобно», врёт систематически.

Теория — на сайте курса: `ch05/03-sampling.html`. Задачи «собери код из строк» (1) остались на сайте.

### Параметры «совокупности» · разминка

Считая все доставки с временем 1–299 минут совокупностью, сохраните её среднее `mu` и стандартное отклонение `sigma` (оба — 2 знака).

*Подсказки (2): `dsc.hint('population')`*

In [ ]:
%%task ch05/03-sampling/population
import pandas as pd
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"]

### Одна выборка · разминка

Возьмите из `t` выборку из 50 доставок (`t.sample(50, random_state=7)`) и сохраните её среднее в `m50` (2 знака), а в `error` — на сколько оно отличается от среднего всей совокупности (по модулю, 2 знака).

*Подсказки (2): `dsc.hint('ch05/03-sampling/one-sample')`*

In [ ]:
%%task ch05/03-sampling/one-sample
import pandas as pd
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"]

### Формула против симуляции · база

Возьмите 3 000 выборок по 30 доставок (`rng = np.random.default_rng(2)`) и сохраните массив их средних в `means`. Сохраните `se_sim` — стандартное отклонение средних и `se_formula` — σ/√30 (оба — 2 знака; σ — `t.std()`).

*Подсказки (2): `dsc.hint('se-check')`*

In [ ]:
%%task ch05/03-sampling/se-check
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"].to_numpy()
rng = np.random.default_rng(2)

### Функция стандартной ошибки · база

Напишите функцию `se(sigma, n)` — стандартная ошибка среднего. Сохраните в `ses` список стандартных ошибок для σ = 11 и n = 25, 100, 400 (по 2 знака).

*Подсказки (2): `dsc.hint('se-fn')`*

In [ ]:
%%task ch05/03-sampling/se-fn
import math

def se(sigma, n):
    ...

ses = []

### Сколько нужно данных · база

Руководитель хочет знать среднее время доставки с точностью ±1 минута, то есть чтобы стандартная ошибка была не больше 0.5 минуты. Сколько доставок для этого нужно, если σ = 11 минут? Сохраните ответ в `n_needed` — целое число, округлённое вверх.

*Подсказки (2): `dsc.hint('n-needed')`*

In [ ]:
%%task ch05/03-sampling/n-needed
import math
sigma = 11

### Анкета на чеке · база

Проверим игрушечный пример на настоящей базе гостей. Сохраните `true_mean` — среднее `visits_90d` по всем гостям, `hall_mean` — среднее в опросе по 500 случайным чекам (`g.sample(500, weights="visits_90d", replace=True, random_state=5)`), `bias` — разницу `hall_mean − true_mean` и `theory` — чего ждать от отбора по чекам по формуле из игрушечного примера: сумма квадратов визитов, делённая на сумму визитов. Все — 2 знака.

*Подсказки (3): `dsc.hint('hall-survey')`*

In [ ]:
%%task ch05/03-sampling/hall-survey
import pandas as pd
g = pd.read_csv("guests_churn.csv")
v = g["visits_90d"]

### Приложение против обзвона · посложнее

Маркетинг хочет оценить долю ушедших гостей (`churned`) по опросу в приложении: ответили все 1 059 пользователей с `app == 1`. Аналитик предлагает другое: обзвонить 150 случайных гостей из всей базы (`g.sample(150, random_state=7)`). Сохраните: `true_rate` — долю ушедших среди всех гостей, `app_rate` — среди пользователей приложения, `call_rate` — в обзвоне, `err_app` и `err_call` — модули отклонений от `true_rate`, `se_call` — стандартную ошибку доли для обзвона (по `call_rate` и n = 150). Все — 3 знака. В `better` запишите строку `"app"` или `"call"` — чья ошибка меньше.

*Подсказки (3): `dsc.hint('app-survey')`*

In [ ]:
%%task ch05/03-sampling/app-survey
import pandas as pd
g = pd.read_csv("guests_churn.csv")

### ЦПТ своими глазами · посложнее

Нарисуйте две гистограммы рядом: слева — расстояния доставки `distance_km` (сильно скошены), справа — средние 3 000 выборок по 50 расстояний (`rng = np.random.default_rng(4)`). Сохраните `skew_data` и `skew_means` — асимметрию данных и средних (2 знака). Асимметрия — `pd.Series(arr).skew()` — мера скошенности: 0 — симметрично, больше 0 — длинный хвост вправо, меньше 0 — влево.

*Подсказки (2): `dsc.hint('clt')`*

In [ ]:
%%task ch05/03-sampling/clt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
d = pd.read_csv("deliveries.csv")
km = d["distance_km"].to_numpy()
rng = np.random.default_rng(4)

### Ошибка доли · посложнее

Доля опозданий p = 0.24. Сохраните `se_100` — стандартную ошибку доли при n = 100 по формуле (4 знака). Проверьте симуляцией: 5 000 выборок по 100 доставок с вероятностью опоздания 0.24 (`rng = np.random.default_rng(6)`, таблица 5 000 × 100), разброс их долей — `se_sim` (4 знака). И ответьте на вопрос руководителя: сколько доставок нужно, чтобы SE доли была не больше 2 п.п.? Сохраните `n_2pp` — целое число, округлённое вверх.

*Подсказки (3): `dsc.hint('se-share')`*

In [ ]:
%%task ch05/03-sampling/se-share
import math
import numpy as np
rng = np.random.default_rng(6)

### 95 % в пределах двух ошибок · со звёздочкой

Возьмите 4 000 выборок по 40 доставок (время 1–299 минут, `rng = np.random.default_rng(9)`). Для каждой посчитайте среднее. Сохраните в `share_within` долю выборок, у которых среднее попало в пределы «настоящее среднее ± 1.96 · SE» (SE = σ/√40, σ и среднее — по всей совокупности). Ожидаем около 0.95.

*Подсказки (2): `dsc.hint('coverage')` · задача по желанию*

In [ ]:
%%task ch05/03-sampling/coverage
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"].to_numpy()
rng = np.random.default_rng(9)

## Урок 5. Доверительные интервалы

«Средняя доставка в KZN-2 — 37.8 минуты» звучит точно, но по выборке из 272 доставок это лишь оценка. Доверительный интервал честно говорит, где, скорее всего, лежит настоящее значение: «от 36.6 до 39.0 минут». Строим интервалы для среднего, для доли, для медианы — и для разницы двух групп.

Теория — на сайте курса: `ch05/04-confidence.html`. Задачи «собери код из строк» (1) остались на сайте.

### Интервал для среднего · разминка

Постройте 95 %-й ДИ для среднего времени доставки кофейни `PRM-1` (время 1–299 минут) по формуле «среднее ± 1.96 · SE». Сохраните `lo` и `hi` (2 знака).

*Подсказки (2): `dsc.hint('ci-mean')`*

In [ ]:
%%task ch05/04-confidence/ci-mean
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
p = ok.loc[ok["shop_id"] == "PRM-1", "actual_min"]

### Через scipy · разминка

Постройте тот же интервал для `PRM-1` функцией `stats.t.interval` и сохраните `lo_t`, `hi_t` (2 знака), а в `t_mult` — множитель t для этой выборки (`stats.t.ppf(0.975, n − 1)`, 3 знака).

*Подсказки (2): `dsc.hint('t-interval')`*

In [ ]:
%%task ch05/04-confidence/t-interval
import numpy as np
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
p = ok.loc[ok["shop_id"] == "PRM-1", "actual_min"]

### Интервалы по курьерам · база

Соберите таблицу `ci` по типу курьера со столбцами `n`, `late` (доля опозданий), `lo`, `hi` (границы интервала Уилсона, 95 %). Все доли — 3 знака.

*Подсказки (3): `dsc.hint('courier-ci')`*

In [ ]:
%%task ch05/04-confidence/courier-ci
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]

### Уровень доверия и ширина · база

Для времени доставки кофейни `UFA-1` (1–299 минут) постройте t-интервалы на уровнях 90 %, 95 % и 99 %. Сохраните в `widths` словарь «уровень → ширина интервала» (ключи 0.90, 0.95, 0.99; ширина — 2 знака).

*Подсказки (2): `dsc.hint('widths')`*

In [ ]:
%%task ch05/04-confidence/widths
import numpy as np
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
u = d.loc[d["actual_min"].between(1, 299) & (d["shop_id"] == "UFA-1"), "actual_min"]

### Бутстрэп для медианы · база

Постройте 95 %-й бутстрэп-интервал для **медианы** времени доставки вело-курьеров (1–299 минут): 5 000 перевыборок с возвращением, `rng = np.random.default_rng(12)`. Сохраните `med`, `lo`, `hi`.

*Подсказки (2): `dsc.hint('bootstrap')`*

In [ ]:
%%task ch05/04-confidence/bootstrap
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299) & (d["courier"] == "вело"), "actual_min"].to_numpy()
rng = np.random.default_rng(12)

### Интервалы на графике · посложнее

Нарисуйте долю опозданий по кофейням точками с «усами» 95 %-го интервала Уилсона (`ax.errorbar`), кофейни отсортированы по доле. Подпись оси Y — «доля опозданий».

*Подсказки (2): `dsc.hint('errorbars')`*

In [ ]:
%%task ch05/04-confidence/errorbars
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
g = ok.groupby("shop_id")["late"].agg(["sum", "size", "mean"]).sort_values("mean")

### Интервал для разницы · посложнее

Постройте 95 %-й ДИ для разницы долей опозданий в часы пик (12, 13, 18, 19) и в остальные часы. Сохраните `diff` (пик минус остальные), `lo`, `hi` (все — в процентных пунктах, 1 знак) и `zero_inside` — попадает ли ноль в интервал.

*Подсказки (2): `dsc.hint('diff-ci')`*

In [ ]:
%%task ch05/04-confidence/diff-ci
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
peak = ok["created"].dt.hour.isin([12, 13, 18, 19])

### Когда 95 % — не 95 % · со звёздочкой

Проверьте, сколько на самом деле накрывает 95 %-й t-интервал на **маленьких** выборках из скошенных данных. «Совокупность» — все времена доставки 1–299 минут. Сделайте 4 000 выборок по 8 доставок (`rng = np.random.default_rng(13)`), для каждой постройте t-интервал и сохраните в `coverage` долю интервалов, накрывших настоящее среднее совокупности (3 знака).

*Подсказки (2): `dsc.hint('coverage-small')` · задача по желанию*

In [ ]:
%%task ch05/04-confidence/coverage-small
import numpy as np
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"].to_numpy()
mu = t.mean()
rng = np.random.default_rng(13)

## Урок 6. Проверка гипотез: логика и p-value

В TMN-1 опаздывают в 28 % доставок, в PRM-1 — в 23 %. Тюмень работает хуже — или это случайность? Проверка гипотез отвечает на такие вопросы по единой схеме. Разбираем её на перестановочном тесте — самом наглядном: никаких формул, только перемешивание данных.

Теория — на сайте курса: `ch05/05-hypotheses.html`. Задачи «собери код из строк» (1) остались на сайте.

### Наблюдаемая разница · разминка

Сохраните в `observed` разницу долей опозданий в снег и в остальную погоду (снег минус остальное), в процентных пунктах, 1 знак.

*Подсказки (2): `dsc.hint('observed')`*

In [ ]:
%%task ch05/05-hypotheses/observed
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
snow = ok["weather"] == "снег"

### Решение по p-value · разминка

Напишите функцию `decision(p, alpha=0.05)`, которая возвращает строку `"отвергаем H0"`, если p меньше α, и `"не отвергаем H0"` иначе. Сохраните в `answers` список решений для p = 0.001, 0.049, 0.05, 0.3.

*Подсказки (2): `dsc.hint('ch05/05-hypotheses/decision')`*

In [ ]:
%%task ch05/05-hypotheses/decision
def decision(p, alpha=0.05):
    ...

answers = []

### Перестановочный тест: снег · база

Проведите перестановочный тест для разницы долей опозданий «снег — остальная погода»: 3 000 перемешиваний, `rng = np.random.default_rng(0)`. Сохраните `diffs` (массив разниц) и `p_value` — двустороннее p-value.

*Подсказки (2): `dsc.hint('perm-snow')`*

In [ ]:
%%task ch05/05-hypotheses/perm-snow
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
late = (ok["actual_min"] > ok["promised_min"]).to_numpy()
snow = (ok["weather"] == "снег").to_numpy()
rng = np.random.default_rng(0)

### Одно- и двусторонний · база

Для TMN-1 против PRM-1 (разница долей опозданий, TMN-1 минус PRM-1) проведите перестановочный тест на 5 000 перемешиваний (`rng = np.random.default_rng(1)`). Сохраните `p_two` — двустороннее p-value и `p_one` — одностороннее для H₁ «в TMN-1 опаздывают чаще». В `same_verdict` — `True`, если при α = 0.05 оба теста дают одинаковое решение.

*Подсказки (2): `dsc.hint('one-sided')`*

In [ ]:
%%task ch05/05-hypotheses/one-sided
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
x = ok[ok["shop_id"].isin(["TMN-1", "PRM-1"])]
late = (x["actual_min"] > x["promised_min"]).to_numpy()
grp = (x["shop_id"] == "TMN-1").to_numpy()
rng = np.random.default_rng(1)

### Ложные тревоги · база

Смоделируйте 2 000 «A/A-экспериментов»: в каждом две группы по 250 доставок с **одинаковой** вероятностью опоздания 0.24 (`rng = np.random.default_rng(3)`). Для каждого посчитайте p-value z-теста для долей (раздел «Быстрая формула: z-тест для двух долей») и сохраните в `false_rate` долю экспериментов с p < 0.05.

*Подсказки (2): `dsc.hint('aa')`*

In [ ]:
%%task ch05/05-hypotheses/aa
import numpy as np
from scipy import stats
rng = np.random.default_rng(3)
n = 250

### Мощность · посложнее

Пусть эффект **есть**: в группе A опаздывают 24 %, в группе B — 30 %. Оцените симуляцией мощность z-теста (долю экспериментов, где p < 0.05) при n = 200 и n = 800 в каждой группе: по 2 000 экспериментов, `rng = np.random.default_rng(4)`. Сохраните `power_200` и `power_800`.

*Подсказки (2): `dsc.hint('power')`*

In [ ]:
%%task ch05/05-hypotheses/power
import numpy as np
from scipy import stats
rng = np.random.default_rng(4)

def power(n, pa=0.24, pb=0.30, sims=2000):
    ...

power_200 = power(200)
power_800 = power(800)

### Функция перестановочного теста · посложнее

Напишите `perm_test(a, b, n_perm=2000, seed=0)` — двусторонний перестановочный тест для разницы **средних** двух массивов чисел. Функция возвращает пару `(разница средних a − b, p-value)`. Проверьте её на времени доставки авто- и вело-курьеров: результат сохраните в `res`.

*Подсказки (2): `dsc.hint('perm-fn')`*

In [ ]:
%%task ch05/05-hypotheses/perm-fn
import numpy as np
import pandas as pd

def perm_test(a, b, n_perm=2000, seed=0):
    ...

d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
auto = ok.loc[ok["courier"] == "авто", "actual_min"].to_numpy()
velo = ok.loc[ok["courier"] == "вело", "actual_min"].to_numpy()
res = perm_test(auto, velo)
res

### Двадцать сегментов · со звёздочкой

Аналитик сравнил группы A и B **в 20 разных сегментах** (города, дни, каналы), хотя на самом деле эффекта нет нигде. Смоделируйте это: 2 000 «исследований», в каждом 20 A/A-сравнений по 300 доставок в группе с p = 0.24 (`rng = np.random.default_rng(5)`). Сохраните в `p_any` долю исследований, где **хотя бы одно** из 20 сравнений дало p < 0.05, и в `p_formula` — то же по формуле `1 − 0.95²⁰` (3 знака).

*Подсказки (2): `dsc.hint('many')` · задача по желанию*

In [ ]:
%%task ch05/05-hypotheses/many
import numpy as np
from scipy import stats
rng = np.random.default_rng(5)

## Урок 7. Статистические тесты: какой выбрать

Перестановочный тест универсален, но медленный и требует писать цикл. Для типовых вопросов есть готовые тесты в scipy и statsmodels — одна строка вместо цикла. Разбираем шесть самых нужных, учимся выбирать тест по вопросу и типу данных — и решать, когда тест может быть односторонним (`alternative=`).

Теория — на сайте курса: `ch05/06-tests.html`. Задачи «собери код из строк» (1) остались на сайте.

### Пешие против вело · разминка

Сравните t-тестом Уэлча среднее время доставки пеших и вело-курьеров (1–299 минут). Сохраните `t_stat` (4 знака) и `p_value` — без округления: оно меньше 0.0001, и округление превратило бы его в ноль.

*Подсказки (2): `dsc.hint('welch')`*

In [ ]:
%%task ch05/06-tests/welch
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]

### Обещание 30 минут · разминка

Проверьте, укладываемся ли мы в среднем в обещание 30 минут: t-тест для одной выборки по доставкам с `promised_min == 30`. Сохраните `mean_30` (среднее, 1 знак) и `p_value` (без округления или с 4 знаками — проверка примет оба). В `beyond` — `True`, если среднее **больше** 30 и при этом p < 0.05.

*Подсказки (2): `dsc.hint('ch05/06-tests/one-sample')`*

In [ ]:
%%task ch05/06-tests/one-sample
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]

### До и после · база

Посчитайте для каждой кофейни долю опозданий в октябре и в ноябре (таблица `m`: строки — кофейни, столбцы — месяцы 10 и 11). Проведите парный t-тест «ноябрь против октября» и сохраните `mean_change` — среднее изменение доли по кофейням в п.п. (1 знак) и `p_value` (3 знака).

*Подсказки (2): `dsc.hint('ch05/06-tests/paired')`*

In [ ]:
%%task ch05/06-tests/paired
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
ok["month"] = ok["created"].dt.month

### Погода и опоздания · база

Проверьте хи-квадратом, связана ли погода с опозданием. Сохраните таблицу сопряжённости в `ct` (строки — погода, столбцы — `late`), а в `chi2_p` — p-value. В `max_gap_weather` — вид погоды, где наблюдаемое число опозданий сильнее всего **превышает** ожидаемое при независимости.

*Подсказки (2): `dsc.hint('chi2')`*

In [ ]:
%%task ch05/06-tests/chi2
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]

### Оценки: Манна — Уитни · база

Сравните оценки гостей при доставке в снег и в остальную погоду тестом Манна — Уитни. Сохраните `med_snow`, `med_other` — медианы оценок и `p_value` (4 знака).

*Подсказки (2): `dsc.hint('mwu')`*

In [ ]:
%%task ch05/06-tests/mwu
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
r = ok.dropna(subset=["rating"])

### Дождь в Казани: дольше ли · база

План анализа записан заранее: «добавляем 5 минут к обещанию в дождь, если в дождь доставки дольше, чем в ясную погоду; α = 0.05». Проверьте его для кофейни KZN-2 тестом Уэлча (время 1–299 минут). Сохраните p-value трёх вариантов: `p_two` (двусторонний), `p_greater` (H₁: в дождь дольше) и `p_less` (H₁: в дождь быстрее) — все 4 знака. В `add_5min` — `True` или `False`: решение по тесту, который соответствует плану.

*Подсказки (3): `dsc.hint('alt-rain')`*

In [ ]:
%%task ch05/06-tests/alt-rain
import pandas as pd
from scipy import stats
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299) & (d["shop_id"] == "KZN-2")]

### Какой тест · посложнее

Напишите функцию `choose_test(kind, groups, paired=False, skewed=False)`, которая по описанию задачи возвращает название теста:

- `kind="share"`, `groups=2` → `"proportions_ztest"`;
- `kind="category"` (любое число групп) → `"chi2_contingency"`;
- `kind="mean"`, `groups=1` → `"ttest_1samp"`;
- `kind="mean"`, `groups=2`, `paired=True` → `"ttest_rel"`;
- `kind="mean"`, `groups=2`, `skewed=True` → `"mannwhitneyu"`;
- `kind="mean"`, `groups=2` (иначе) → `"ttest_ind"`.

*Подсказки (2): `dsc.hint('choose')`*

In [ ]:
%%task ch05/06-tests/choose
def choose_test(kind, groups, paired=False, skewed=False):
    ...

choose_test("mean", 2)

### Формула и перемешивание согласны · посложнее

Сравните долю опозданий в KZN-2 и KZN-1 двумя способами: z-тестом для долей (`p_z`) и перестановочным тестом на 4 000 перемешиваний (`p_perm`, `rng = np.random.default_rng(2)`). Оба p-value должны быть близки. Сохраните ещё `diff_pp` — разницу долей в п.п. (1 знак).

*Подсказки (2): `dsc.hint('agree')`*

In [ ]:
%%task ch05/06-tests/agree
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
x = ok[ok["shop_id"].isin(["KZN-2", "KZN-1"])]
late = (x["actual_min"] > x["promised_min"]).to_numpy()
grp = (x["shop_id"] == "KZN-2").to_numpy()
rng = np.random.default_rng(2)

### Фраза для отчёта · со звёздочкой

Напишите функцию `compare_shares(late_a, late_b)`: принимает два массива `True/False` (опоздания в группах A и B) и возвращает строку вида

`"A: 30.0 %, B: 20.0 %, разница 10.0 п.п. (95 % ДИ -1.9…21.9), p = 0.1025"`

— доли и разница с 1 знаком, интервал разницы — по формуле из урока «Доверительные интервалы», p — z-тест для долей, 4 знака. Между границами интервала — символ многоточия «…», а не три точки. Образец выше — для выдуманных групп 30 из 100 и 20 из 100. Проверьте функцию на KZN-2 (A) и KZN-1 (B): результат — в `phrase`.

*Подсказки (2): `dsc.hint('ch05/06-tests/report')` · задача по желанию*

In [ ]:
%%task ch05/06-tests/report
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

def compare_shares(late_a, late_b):
    ...

d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
late = ok["actual_min"] > ok["promised_min"]
phrase = compare_shares(late[ok["shop_id"] == "KZN-2"].to_numpy(), late[ok["shop_id"] == "KZN-1"].to_numpy())
phrase

## Урок 8. A/B-тест: от дизайна до решения

Срезы и корреляции находят связи, но не доказывают причины. Доказать, что изменение работает, может эксперимент: часть гостей видит новое, часть — старое, и всё остальное у групп одинаково. Проходим A/B-тест целиком: дизайн и размер выборки, проверка данных, анализ главной метрики, денег и решение.

Теория — на сайте курса: `ch05/07-ab.html`. Задачи «собери код из строк» (1) остались на сайте.

### Проверка разбиения · разминка

Сохраните в `sizes` число гостей в группах (Series с индексом A, B) и в `srm_p` — p-value проверки на SRM (`stats.chisquare`). В `srm_ok` — `True`, если p ≥ 0.01.

*Подсказки (2): `dsc.hint('srm')`*

In [ ]:
%%task ch05/07-ab/srm
import pandas as pd
from scipy import stats
ab = pd.read_csv("ab_push.csv")

### Конверсии групп · разминка

Сохраните `conv` — словарь «группа → конверсия в заказ» (доля `ordered`, 4 знака) и `lift_pp` — разницу B − A в процентных пунктах (2 знака).

*Подсказки (2): `dsc.hint('ch05/07-ab/conv')`*

In [ ]:
%%task ch05/07-ab/conv
import pandas as pd
ab = pd.read_csv("ab_push.csv")

### Сколько гостей нужно · база

Маркетинг хочет проверить новый пуш, который, по их оценке, поднимет конверсию с 9 % до 10 %. Сохраните `n_needed` — сколько гостей нужно **в каждой группе** (мощность 80 %, α = 0.05, округление вверх), и `n_double_mde` — сколько нужно, если искать рост до 11 %. В `ratio` — во сколько раз первое больше второго (1 знак).

*Подсказки (2): `dsc.hint('sample-size')`*

In [ ]:
%%task ch05/07-ab/sample-size
import math
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

### Значима ли разница · база

Проверьте z-тестом, различается ли конверсия групп. Сохраните `p_value` (4 знака) и 95 %-й ДИ разницы B − A в п.п.: `ci_lo`, `ci_hi` (2 знака).

*Подсказки (2): `dsc.hint('ztest')`*

In [ ]:
%%task ch05/07-ab/ztest
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
ab = pd.read_csv("ab_push.csv")

### Откуда рост выручки · база

Разложите рост выручки на гостя. Сохраните:

- `arpu` — словарь «группа → выручка на гостя» (1 знак);
- `per_buyer` — словарь «группа → выручка на покупателя» (среди `ordered == 1`, 1 знак);
- `p_buyer` — p-value t-теста Уэлча для выручки на покупателя B против A (3 знака).

*Подсказки (2): `dsc.hint('per-buyer')`*

In [ ]:
%%task ch05/07-ab/per-buyer
import pandas as pd
from scipy import stats
ab = pd.read_csv("ab_push.csv")

### Бутстрэп выручки · посложнее

Постройте 95 %-й бутстрэп-интервал разницы выручки на гостя (B − A): 3 000 перевыборок обеих групп, `rng = np.random.default_rng(1)`. Сохраните `diff` (1 знак), `lo`, `hi`, и `zero_inside`.

*Подсказки (2): `dsc.hint('boot-arpu')`*

In [ ]:
%%task ch05/07-ab/boot-arpu
import numpy as np
import pandas as pd
ab = pd.read_csv("ab_push.csv")
a = ab.loc[ab["group"] == "A", "revenue"].to_numpy()
b = ab.loc[ab["group"] == "B", "revenue"].to_numpy()
rng = np.random.default_rng(1)

### Отчёт по тесту · посложнее

Напишите `ab_report(df, metric)`: принимает таблицу теста со столбцом `group` (A/B) и названием столбца-метрики из 0/1, возвращает словарь:

- `"n_a"`, `"n_b"` — размеры групп;
- `"srm_p"` — p-value проверки SRM;
- `"a"`, `"b"` — доли метрики в группах (4 знака);
- `"diff_pp"` — разница B − A в п.п. (2 знака);
- `"p"` — p-value z-теста (4 знака);
- `"significant"` — `True`, если p < 0.05 **и** srm_p ≥ 0.01.

Сохраните результат для `ab_push.csv` и метрики `ordered` в `report`.

*Подсказки (2): `dsc.hint('ab-report')`*

In [ ]:
%%task ch05/07-ab/ab-report
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

def ab_report(df, metric):
    ...

ab = pd.read_csv("ab_push.csv")
report = ab_report(ab, "ordered")
report

### Что мы вообще могли увидеть · со звёздочкой

Тест уже прошёл: по 3 000 гостей в группе, базовая конверсия 9 %. Какой **минимальный** рост мы могли надёжно (мощность 80 %, α = 0.05) обнаружить? Найдите размер эффекта `h` через `solve_power(nobs1=3000, ...)` и переведите его в конверсию группы B: `p_b = sin((h + 2·asin(√p_a)) / 2)²`. Сохраните `mde_pp` — разницу p_b − p_a в п.п. (2 знака).

*Подсказки (2): `dsc.hint('mde')` · задача по желанию*

In [ ]:
%%task ch05/07-ab/mde
import math
from statsmodels.stats.power import NormalIndPower
p_a = 0.09

## Урок 9. Ловушки статистики

Статистические тесты честны, пока их используют по правилам. Нарушения бывают незаметными: подглядели в результаты раньше срока, проверили двадцать сегментов, обрадовались «значимой» разнице в 0.1 %. Разбираем пять самых частых ловушек — каждую на симуляции, чтобы увидеть масштаб проблемы своими глазами.

Теория — на сайте курса: `ch05/08-pitfalls.html`. Задачи «собери код из строк» (1) остались на сайте.

### Порог Бонферрони · разминка

Аналитик проверил эффект теста по 8 метрикам. Сохраните в `threshold` порог Бонферрони для α = 0.05, а в `significant` — сколько из p-values `[0.001, 0.004, 0.008, 0.02, 0.03, 0.04, 0.3, 0.6]` проходят этот порог.

*Подсказки (2): `dsc.hint('bonferroni')`*

In [ ]:
%%task ch05/08-pitfalls/bonferroni
pvals = [0.001, 0.004, 0.008, 0.02, 0.03, 0.04, 0.3, 0.6]

### Поправка Холма · разминка

Примените поправку Холма к тем же p-values. Сохраните `reject` — **список** `True/False` и `n_holm` — сколько гипотез отвергнуто.

*Подсказки (2): `dsc.hint('holm')`*

In [ ]:
%%task ch05/08-pitfalls/holm
from statsmodels.stats.multitest import multipletests
pvals = [0.001, 0.004, 0.008, 0.02, 0.03, 0.04, 0.3, 0.6]

### Города после теста · база

Посчитайте p-value z-теста конверсии (B против A) отдельно для каждого города в `ab_push.csv`. Сохраните словарь `p_city` (город → p, 4 знака), список `raw_sig` — города, где p < 0.05 **без** поправки, и `holm_sig` — города, значимые после поправки Холма (оба списка — по алфавиту).

*Подсказки (2): `dsc.hint('segments')`*

In [ ]:
%%task ch05/08-pitfalls/segments
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
ab = pd.read_csv("ab_push.csv")

### Цена подглядывания · база

Повторите симуляцию подглядывания из урока, но с **7** проверками (раз в день в течение недели, по 200 гостей в группе в день, конверсия 10 % в обеих группах), 1 000 симуляций, `rng = np.random.default_rng(2)`. Сохраните `peek_rate` — долю симуляций, где хотя бы одна из 7 проверок дала p < 0.05, и `final_rate` — долю, где p < 0.05 только на **последней** проверке (правильный способ).

*Подсказки (2): `dsc.hint('peeking')`*

In [ ]:
%%task ch05/08-pitfalls/peeking
import numpy as np
from scipy import stats
rng = np.random.default_rng(2)
days, per_day, sims = 7, 200, 1000

### Значимо, но мелко · база

Тест на огромной аудитории: в группе A 2 000 000 гостей и 400 000 заказов, в группе B — 2 000 000 гостей и 402 400 заказов. Сохраните `p_value`, `diff_pp` (разница в п.п., 2 знака), `rel_pct` (относительный рост, %, 1 знак) и `worth_it` — `True`, если относительный рост не меньше 2 % (порог полезности, договорённый с бизнесом) **и** p < 0.05.

*Подсказки (2): `dsc.hint('practical')`*

In [ ]:
%%task ch05/08-pitfalls/practical
from statsmodels.stats.proportion import proportions_ztest

### Подглядывать можно, но строже · посложнее

Иногда тест всё-таки нужно иметь право остановить досрочно. Простое решение — поправка на число проверок: на каждой из 7 проверок требовать p < 0.05 / 7. Это та же поправка Бонферрони, что для многих метрик: подглядывание — тоже множественные сравнения, только во времени. Повторите симуляцию подглядывания из задачи «Цена подглядывания» (те же параметры и `rng = np.random.default_rng(2)`) и сохраните `strict_rate` — долю A/A-тестов, где хотя бы одна проверка прошла строгий порог.

*Подсказки (2): `dsc.hint('strict-peek')`*

In [ ]:
%%task ch05/08-pitfalls/strict-peek
import numpy as np
from scipy import stats
rng = np.random.default_rng(2)
days, per_day, sims = 7, 200, 1000

### Регрессия к среднему · посложнее

Смоделируйте 50 кофеен с **одинаковой** вероятностью опоздания 0.24, по 80 доставок в месяц, два месяца (`rng = np.random.default_rng(3)`). Найдите 5 худших в первом месяце. Сохраните `worst_m1` — их среднюю долю опозданий в первом месяце, `worst_m2` — во втором (обе — 3 знака), и `improved` — `True`, если во втором месяце они «улучшились».

*Подсказки (2): `dsc.hint('regression')`*

In [ ]:
%%task ch05/08-pitfalls/regression
import numpy as np
rng = np.random.default_rng(3)
shops, n, p = 50, 80, 0.24

### Поправка спасает · со звёздочкой

Повторите «двадцать сегментов» из урока «Проверка гипотез: логика и p-value» (2 000 исследований × 20 A/A-сравнений, по 300 доставок в группе, p = 0.24, `rng = np.random.default_rng(5)`, доли через `rng.binomial`), но теперь посмотрите, что сделает поправка Холма. Сохраните:

- `p_any_raw` — долю исследований хотя бы с одной «значимой» находкой без поправки;
- `p_any_holm` — то же с поправкой Холма;
- `same` — `True`, если на первых 100 исследованиях ваш способ для Холма совпал с `multipletests(row, method="holm")`.

Вызывать `multipletests` 2 000 раз не нужно: в браузере это пара минут. Подумайте, когда Холм отклоняет **хотя бы одну** гипотезу.

*Подсказки (3): `dsc.hint('holm-rescue')` · задача по желанию*

In [ ]:
%%task ch05/08-pitfalls/holm-rescue
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
rng = np.random.default_rng(5)
n = 300

## Урок 10. Проект: новое обещание доставки

Финал главы. В главе 4 мы выяснили, что пешим курьерам обещают нереальные 30 минут. Бизнес провёл эксперимент: половине гостей в пешей зоне стали обещать 40 минут. Помогло ли это — и не испортило ли продажи? Анализируем A/B-тест целиком: от проверки дизайна до решения.

Теория — на сайте курса: `ch05/09-project.html`.

### Шаг 1. Проверка эксперимента · база

Загрузите `ab_promise.csv` (`ts` — датой) в `ab`. Проверьте, что эксперимент устроен как задумано:

- `srm_p` — p-value проверки SRM по числу сессий в группах;
- `promise_ok` — `True`, если в зоне до 2 км **все** сессии группы A видели обещание 30 минут, а группы B — 40 минут;
- `far_same` — `True`, если в зоне от 2 км набор обещаний у групп одинаковый (множества значений `promise_min` совпадают).

*Подсказки (2): `dsc.hint('design')`*

In [ ]:
%%task ch05/09-project/design
import pandas as pd
from scipy import stats

### Шаг 2. Главная метрика: опоздания · база

Главная метрика — доля опозданий среди **заказов** (`ordered == 1`). Сохраните `late_a`, `late_b` (доли, 4 знака), `diff_pp` (B − A в п.п., 1 знак), `ci` — кортеж (нижняя, верхняя граница 95 % ДИ разницы в п.п., 1 знак) и `p_late` — p-value z-теста (без округления: оно меньше 0.0001).

*Подсказки (2): `dsc.hint('late')`*

In [ ]:
%%task ch05/09-project/late
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
# ab — вся выгрузка, orders — только сессии с заказом

### Шаг 3. Конверсия: откуда могут уйти деньги · база

Более долгое обещание может отпугнуть гостей. Сравните конверсию сессий в заказ (`ordered`): в целом, в зоне до 2 км и в зоне от 2 км. Сохраните p-values z-тестов `p_all`, `p_short`, `p_far` и разницы B − A в п.п. (2 знака): `d_all`, `d_short`, `d_far`.

*Подсказки (2): `dsc.hint('ch05/09-project/conversion')`*

In [ ]:
%%task ch05/09-project/conversion
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

### Шаг 4. Оценки гостей · база

Сравните оценки доставки (среди заказов с оценкой) в группах: средние `rating_a`, `rating_b` (2 знака), доли «не пятёрок» `low_a`, `low_b` (оценка ≤ 4, 3 знака) и `p_rating` — p-value теста Манна — Уитни.

*Подсказки (2): `dsc.hint('ch05/09-project/ratings')`*

In [ ]:
%%task ch05/09-project/ratings
import pandas as pd
from scipy import stats

### Шаг 5. Деньги на сессию · посложнее

Выручка на сессию (`amount`, у сессий без заказа — 0) — итог «меньше опозданий» против «меньше заказов». Сохраните `rev_a`, `rev_b` (1 знак), `p_rev` — p-value t-теста Уэлча и 95 %-й бутстрэп-интервал разницы B − A: `rev_lo`, `rev_hi` (3 000 перевыборок, `rng = np.random.default_rng(0)`).

*Подсказки (2): `dsc.hint('ch05/09-project/money')`*

In [ ]:
%%task ch05/09-project/money
import numpy as np
from scipy import stats
rng = np.random.default_rng(0)

### Шаг 6. Эффект по городам · посложнее

Проверьте, снижаются ли опоздания во всех городах. Для каждого города посчитайте долю опозданий среди заказов в A и B, разницу в п.п. и p-value z-теста. Соберите таблицу `by_city` со столбцами `late_a`, `late_b`, `diff_pp`, `p`, `p_holm` (p после поправки Холма). В `all_cities` — `True`, если после поправки эффект значим во всех городах. Доли и разницу округлять не обязательно — проверка примет и округлённые (доли — до 3 знаков, разницу — до 1), и точные значения; `p` и `p_holm` оставьте без округления.

*Подсказки (2): `dsc.hint('ch05/09-project/cities')`*

In [ ]:
%%task ch05/09-project/cities
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

### Шаг 7. График для руководителя · посложнее

Нарисуйте два графика рядом: слева — доля опозданий в группах A и B (столбцы, %), справа — конверсия в зоне до 2 км (столбцы, %). На каждом столбце — подпись значения с 1 знаком и «усы» 95 %-го интервала Уилсона (`yerr` в `ax.bar`). Заголовки — выводы (любой непустой текст).

*Подсказки (2): `dsc.hint('chart')`*

In [ ]:
%%task ch05/09-project/chart
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

### Шаг 8. Отчёт и решение · со звёздочкой

Напишите `promise_report(ab)`: принимает выгрузку эксперимента и возвращает словарь:

- `"srm_ok"` — p SRM не меньше 0.01;
- `"late_diff_pp"` и `"late_p"` — изменение доли опозданий среди заказов (B − A, п.п., 1 знак) и p-value;
- `"conv_short_diff_pp"` и `"conv_short_p"` — изменение конверсии в зоне до 2 км (п.п., 2 знака) и p-value;
- `"rev_diff"` и `"rev_p"` — изменение выручки на сессию (₽, 1 знак) и p-value Уэлча;
- `"decision"` — `"внедрять"`, если SRM в порядке, опоздания **значимо снизились** и выручка на сессию **значимо не упала** (не «p < 0.05 и разница < 0»); иначе — `"не внедрять"`.

Функция должна работать на любой части данных — например, только на одном городе.

*Подсказки (2): `dsc.hint('ch05/09-project/decision')` · задача по желанию*

In [ ]:
%%task ch05/09-project/decision
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

def promise_report(ab):
    ...

ab = pd.read_csv("ab_promise.csv", parse_dates=["ts"])
promise_report(ab)

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()